<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Fine Tuning con Trainer</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Trainer abstrae gran parte del ciclo de entrenamiento en Hugging Face y permite centrar el trabajo en datos, configuración y evaluación. Este notebook introduce esa capa de automatización aplicada al ajuste fino de modelos.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Facilita reproducibilidad y reduce código repetitivo en tareas habituales de fine-tuning.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué papel cumple Trainer dentro del flujo de adaptación de un modelo preentrenado.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Automatizar el entrenamiento no elimina la necesidad de entender qué decisiones estás delegando.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Fine-tuning adapta un modelo preentrenado a una tarea específica usando nuevos datos y una configuración controlada.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo y un tokenizer preentrenados desde Hugging Face.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. La cabeza de clasificación se inicializa de nuevo y necesita entrenamiento.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos componentes son los necesarios para realizar el fine-tuning de un modelo de clasificación de secuencias.
El modelo y el tokenizador están preentrenados para inglés, mientras que los textos del ejemplo están en español."""

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Muestra el modelo y el tokenizador cargados para verificación.


In [ ]:
# Bloque 2 — Preparación del dataset
"""Este bloque prepara un dataset mínimo para el fine-tuning.
Este bloque importa "torch" como "t", aunque no lo utiliza directamente para la preparación del dataset.
Este bloque importa "Dataset" de "datasets" para crear el dataset.

Definimos un dataset mínimo con textos y etiquetas para el fine-tuning.
La variable "textos" contiene una lista con los textos de entrada para el fine-tuning.
La variable "etiquetas" contiene una lista de etiquetas correspondientes a cada uno de los textos de entrada.

Definimos el diccionario "diccionario" que contiene los textos y las etiquetas correspondientes.
La variable "diccionario" contiene el mapeo de los textos y sus etiquetas correspondientes.

Definimos la lista "lista_columnas" que contiene los nombres de las columnas que se incluirán en el dataset tokenizado.
La variable "lista_columnas" contiene los nombres de las columnas que se incluirán en el dataset tokenizado.

Definimos la función "def tokenize_batch(batch)" para tokenizar cada lote de textos del dataset.
El parámetro "batch" es un diccionario que contiene los textos asociados a la clave "texto" del diccionario definido previamente.

Esta función retorna la llamada al tokenizador "tokenizador()" con los textos del lote, aplicando padding y truncado.
El argumento "batch["texto"]" contiene los textos del lote asociados a la clave "texto".
El parámetro "padding=" rellena las secuencias hasta la longitud de la más larga del lote.
El parámetro "truncation=" trunca las secuencias que exceden la longitud máxima del modelo.

Aplicamos la clase "Dataset", acompañada del método ".from_dict()", a la variable "dataset" para crear el dataset a partir del diccionario de datos.
El argumento "diccionario" es el diccionario de datos definido previamente.
La variable "dataset" contiene el dataset sin tokenizar, listo para ser procesado con el método ".map()".

Aplicamos el método ".map()" al dataset para tokenizar sus textos.
El argumento "tokenize_batch" es la función que se aplica a cada lote para realizar la tokenización.
El parámetro "batched=" indica que la función se aplicará a lotes de ejemplos en lugar de a ejemplos individuales.
La variable "dataset" contiene ahora el dataset tokenizado, listo para ser formateado con el método ".set_format()".

Aplicamos el método ".set_format()" al dataset para establecer el formato de salida.
El parámetro "type=" indica con el valor "torch" que los datos se devolverán como tensores de PyTorch.
El parámetro "columns=" especifica las columnas que se incluirán en la salida, en este caso, las columnas definidas en la lista "lista_columnas".
La variable "dataset" almacena ahora el dataset tokenizado en un formato compatible con Trainer.
Esto permite utilizar el dataset con el Trainer de Hugging Face."""

import torch as t         
from datasets import Dataset
                                
textos = ["Este producto es excelente.", "La experiencia fue terrible."]    
etiquetas = [1, 0] 
diccionario = {"texto": textos, "label": etiquetas}
lista_columnas = ["input_ids", "attention_mask", "label"]

def tokenize_batch(batch):
    return tokenizador(batch["texto"], padding=True, truncation=True)

dataset = Dataset.from_dict(diccionario)
dataset = dataset.map(tokenize_batch, batched=True)
dataset.set_format(type="torch", columns=lista_columnas)

dataset # Muestra una representación del dataset tokenizado listo para Trainer.


In [ ]:
# Bloque 3 — Configuración de TrainingArguments
"""Este bloque define los parámetros de entrenamiento mediante TrainingArguments.
Este bloque importa "TrainingArguments" de "transformers" para definir los argumentos de entrenamiento.

Aplicamos la clase "TrainingArguments()" a la variable "argumentos" para definir los parámetros de entrenamiento.
El parámetro "output_dir=" especifica el directorio donde se guardarán los resultados del entrenamiento, en este caso, "salida_entrenamiento".
El parámetro "num_train_epochs=" indica el número de épocas para el entrenamiento, en este caso, 1.
El parámetro "per_device_train_batch_size=" define el tamaño de batch por dispositivo, en este caso, 2.
El parámetro "logging_steps=" establece cada cuántos pasos se registrará información de entrenamiento, en este caso, 1.

La variable "argumentos" almacena la configuración del proceso, incluyendo el número de épocas y el tamaño de batch.
Estos argumentos controlan el comportamiento del Trainer durante el fine-tuning."""

from transformers import TrainingArguments

argumentos = TrainingArguments(
    output_dir="salida_entrenamiento",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=1
)

argumentos # Muestra la configuración de los argumentos de entrenamiento.


In [ ]:
# Bloque 4 — Definición del Trainer
"""Este bloque define el Trainer que coordina el proceso de fine-tuning.
Este bloque importa "Trainer" de "transformers" para coordinar el proceso de fine-tuning del modelo.

Aplicamos la clase "Trainer()" a la variable "entrenador" para coordinar el proceso de fine-tuning del modelo.
El parámetro "model=" especifica el modelo que se entrenará, en este caso, "modelo".
El parámetro "args=" recibe los argumentos de entrenamiento, en este caso, "argumentos".
El parámetro "train_dataset=" define el dataset de entrenamiento, en este caso, "dataset".

La variable "entrenador" almacena el objeto Trainer configurado con el modelo, los argumentos y el dataset.
Trainer gestiona automáticamente el cálculo de pérdidas, la optimización y el registro de información de entrenamiento.
Este ejemplo no configura un conjunto de evaluación ni métricas de clasificación."""

from transformers import Trainer

entrenador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=dataset
)

entrenador # Muestra el objeto Trainer configurado.


In [ ]:
# Bloque 5 — Ejecución del entrenamiento
"""Este bloque ejecuta el proceso de entrenamiento mediante el método train.
Aplicamos el método ".train()" a la variable "entrenador" para iniciar el proceso de fine-tuning del modelo.
La variable "resultado_entrenamiento" almacena la información generada durante el fine-tuning.
El ajuste con solo dos ejemplos es ilustrativo y no permite concluir que el modelo generalice."""

resultado_entrenamiento = entrenador.train()

resultado_entrenamiento # Muestra la información del entrenamiento.

In [ ]:
# Bloque 6 — Inferencia posterior al entrenamiento
"""Este bloque realiza inferencia utilizando el modelo ajustado.
Definimos un texto de prueba para observar la salida del modelo.
La variable "entrada_prueba" contiene un texto para realizar inferencia con el modelo.

Asignamos la llamada a la instancia de la clase "tokenizador()" a la variable "ids_prueba" mediante la expresión "tokenizador(entrada_prueba, return_tensors="pt")".
El argumento "entrada_prueba" indica el texto que se desea tokenizar.
El parámetro "return_tensors=" indica el tipo de tensor que se desea obtener, en este caso, "pt" para PyTorch.
La variable "ids_prueba" almacena los tensores tokenizados que se pasan al modelo.

Asignamos la llamada a la instancia de la clase "modelo()" a la variable "salida_prueba" mediante la expresión "modelo(**ids_prueba)".
El argumento "**ids_prueba" indica los tensores tokenizados que se pasan al modelo.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "ids_prueba" y pasar sus elementos como argumentos al modelo.
La variable "salida_prueba" contiene la salida del modelo, incluidos los logits generados tras el fine-tuning.
Esta llamada no realiza una evaluación con etiquetas ni desactiva explícitamente los gradientes o el modo de entrenamiento."""

entrada_prueba = "El servicio fue muy bueno."

ids_prueba = tokenizador(entrada_prueba, return_tensors="pt")

salida_prueba = modelo(**ids_prueba)

ids_prueba, salida_prueba # Muestra los tensores tokenizados y la salida del modelo para la entrada de prueba.


In [ ]:
# Bloque 7 — Resultados finales
"""Este bloque organiza los resultados del proceso de fine-tuning con Trainer.
La variable "resultado" almacena en forma de diccionario la salida del entrenamiento y los logits de la inferencia posterior."""

resultado = {
    "resultado_entrenamiento": resultado_entrenamiento,
    "logits_inferencia": salida_prueba.logits # El atributo ".logits" contiene los logits de la salida del modelo.
}

resultado # Muestra el diccionario que contiene los resultados del entrenamiento y los logits de la inferencia posterior.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo Trainer organiza gran parte del flujo práctico de ajuste fino.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Simplificar la implementación no equivale a simplificar el criterio con el que se entrena.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Usar Trainer como una caja negra sin revisar qué datos, métricas y parámetros están guiando el proceso.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Preparación de los Datasets para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Preparación de los Datasets y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">06_Fine_tuning_con_trainer/02_La_preparacion_de_los_datasets.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">